# Part 2 – CNN for Manufacturing Defect Classification

In this notebook we build a basic CNN model to classify product surface images into four categories:
**normal**, **scratch**, **dent**, and **stain**.

The dataset has 480 images (120 per class).


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)


I0000 00:00:1779037547.302950     613 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779037547.303663     613 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779037547.352034     613 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1779037548.607635     613 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779037548.608025     613 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0


## Setup – paths and settings

Change `DATA_DIR` to wherever you put the `part_2_cnn_computer_vision` folder.


In [2]:
DATA_DIR = 'part_2_cnn_computer_vision'
LABELS_CSV = os.path.join(DATA_DIR, 'labels.csv')

IMG_SIZE = (64, 64)
CLASSES = ['normal', 'scratch', 'dent', 'stain']
NUM_CLASSES = len(CLASSES)

os.makedirs('results', exist_ok=True)
os.makedirs('sample_predictions', exist_ok=True)


## Task 1 – Problem Identification

This is an **image classification** problem. Each image belongs to one of four fixed classes.
We just need to predict which class the whole image belongs to — not detect or segment anything.

Image classification is the right choice here because:
- Every image already has one label (the type of defect or no defect)
- We don't need bounding boxes or pixel-level masks
- A simple CNN that outputs probabilities for 4 classes is enough


## Task 2 – Dataset Exploration


In [3]:
df = pd.read_csv(LABELS_CSV)

print('Total images:', len(df))
print()
print('Images per class:')
print(df['class'].value_counts())


Total images: 480

Images per class:
class
normal     120
scratch    120
dent       120
stain      120
Name: count, dtype: int64


In [4]:
# show a bar chart of class distribution
df['class'].value_counts().plot(kind='bar', color=['steelblue', 'orange', 'green', 'red'])
plt.title('Number of Images per Class')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [5]:
# sample one image from each class and show it
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for i, cls in enumerate(CLASSES):
    sample_row = df[df['class'] == cls].iloc[0]
    img_path = os.path.join(DATA_DIR, sample_row['filename'])
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(cls)
    axes[i].axis('off')

plt.suptitle('One Sample Image from Each Class', fontsize=13)
plt.tight_layout()
plt.show()


In [6]:
# checking dimensions of one image -- assuming all images are the same size
first_img_path = os.path.join(DATA_DIR, df.iloc[0]['filename'])
first_img = Image.open(first_img_path)
print('Image size (W x H):', first_img.size)
print('Image mode:', first_img.mode)
print()
print('Dataset looks balanced -- each class has 120 images, so no imbalance issue here.')


Image size (W x H): (96, 96)
Image mode: RGB

Dataset looks balanced -- each class has 120 images, so no imbalance issue here.


## Task 3 – Image Preprocessing

Steps:
1. Load all images from disk using the labels CSV
2. Resize to 64x64
3. Normalize pixel values to [0, 1]
4. Split into train and test sets (80/20)


In [7]:
images = []
labels = []

for idx, row in df.iterrows():
    img_path = os.path.join(DATA_DIR, row['filename'])
    img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
    img_array = np.array(img)
    images.append(img_array)
    labels.append(row['class'])

images = np.array(images, dtype='float32')
print('Loaded images shape:', images.shape)


Loaded images shape: (480, 64, 64, 3)


In [8]:
# normalizing pixel values -- dividing by 255 brings them to 0-1 range
images = images / 255.0

# convert class names to numbers
label_map = {cls: i for i, cls in enumerate(CLASSES)}
labels_numeric = np.array([label_map[l] for l in labels])

print('Label mapping:', label_map)
print('Labels shape:', labels_numeric.shape)


Label mapping: {'normal': 0, 'scratch': 1, 'dent': 2, 'stain': 3}
Labels shape: (480,)


In [9]:
# splitting into train and test
# not setting random_state so results might be slightly different each run
X_train, X_test, y_train, y_test = train_test_split(
    images, labels_numeric, test_size=0.2
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])


Training samples: 384
Testing samples: 96


### Data Augmentation

We apply basic augmentation during training to help the model not memorize the training images.
Just doing horizontal flip for now — could add more augmentation to improve results.


In [10]:
# basic augmentation layer -- only applied during training
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
])

print('Augmentation layers ready')


Augmentation layers ready


E0000 00:00:1779037549.646494     613 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Task 4 – CNN Model

The model has:
- Two Conv2D layers with ReLU activation
- MaxPooling after each conv layer
- A Flatten layer
- One Dense hidden layer
- Output layer with softmax (for 4-class classification)


In [11]:
model = keras.Sequential([
    # augmentation only applied during training
    data_augmentation,

    # first conv block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    layers.MaxPooling2D((2, 2)),

    # second conv block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # flatten and dense layers
    layers.Flatten(),
    layers.Dense(64, activation='relu'),

    # output layer -- 4 classes, softmax gives probabilities
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Task 5 – Model Training and Evaluation


In [12]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/15


 1/10 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - accuracy: 0.3125 - loss: 1.3733

 2/10 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.2969 - loss: 1.5400

 3/10 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.2812 - loss: 1.5865

 4/10 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.2676 - loss: 1.5913

...

 6/10 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.8937 - loss: 0.3644

 7/10 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.8967 - loss: 0.3618

 8/10 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.8994 - loss: 0.3592

 9/10 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.8990 - loss: 0.3592

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.8958 - loss: 0.3528 - val_accuracy: 0.8701 - val_loss: 0.3155


In [13]:
# plotting training and validation accuracy and loss
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['accuracy'], label='train accuracy')
axes[0].plot(history.history['val_accuracy'], label='val accuracy')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='train loss')
axes[1].plot(history.history['val_loss'], label='val loss')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('results/accuracy_loss_curves.png', dpi=120)
plt.show()
print('Saved to results/accuracy_loss_curves.png')


Saved to results/accuracy_loss_curves.png


In [14]:
# evaluate on the test set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')


Test Accuracy : 0.8854
Test Loss     : 0.3208


In [15]:
# confusion matrix
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=120)
plt.show()
print('Saved to results/confusion_matrix.png')


1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


Saved to results/confusion_matrix.png


In [16]:
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=CLASSES))


Classification Report:
              precision    recall  f1-score   support

      normal       0.96      1.00      0.98        27
     scratch       0.76      0.86      0.81        22
        dent       0.84      0.81      0.82        26
       stain       1.00      0.86      0.92        21

    accuracy                           0.89        96
   macro avg       0.89      0.88      0.88        96
weighted avg       0.89      0.89      0.89        96



In [17]:
# show 10 sample predictions from the test set
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

for i in range(10):
    img = X_test[i]
    true_label = CLASSES[y_test[i]]
    pred_label = CLASSES[y_pred[i]]
    is_correct = (true_label == pred_label)

    axes[i].imshow(img)
    axes[i].set_title(
        f'True: {true_label}\nPred: {pred_label}',
        color='green' if is_correct else 'red',
        fontsize=9
    )
    axes[i].axis('off')

plt.suptitle('Sample Predictions  (green = correct, red = wrong)', fontsize=12)
plt.tight_layout()
plt.savefig('sample_predictions/prediction_outputs.png', dpi=120)
plt.show()
print('Saved to sample_predictions/prediction_outputs.png')


Saved to sample_predictions/prediction_outputs.png
